# Target-Gap Engine
### Sales Forecasting & Gen AI Project

## 1. Imports

In [35]:
import pandas as pd
import numpy as np

In [2]:
from sqlalchemy import create_engine

## 2. Connect to MySQL & Load Data

In [3]:
from urllib.parse import quote_plus

In [4]:
DB_USER = "root"
DB_PASSWORD = quote_plus("@ry@nSh1")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "ai_sales_analysis"

In [5]:
connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)


In [6]:
with engine.connect() as conn:
    sales = pd.read_sql("SELECT * FROM sales_transactions", conn)
    products = pd.read_sql("SELECT product_id, category FROM products", conn)
    targets = pd.read_sql("SELECT * FROM targets", conn)

In [7]:
sales['txn_date'] = pd.to_datetime(sales['txn_date'])
sales['year'] = sales['txn_date'].dt.year
sales['month'] = sales['txn_date'].dt.month
sales = sales.merge(products, on='product_id', how='left')

print("Data loaded. Years available:", sorted(sales['year'].unique()))

Data loaded. Years available: [np.int32(2023), np.int32(2024), np.int32(2025)]


## 3. Set Your Target

Yahan apna desired target aur konsa saal ke liye chahiye, wo daalo.


In [8]:
TARGET_AMOUNT = 400000000
TARGET_YEAR = 2026

## 4. Actual Revenue So Far (Target Year)


In [9]:
actual_this_year = sales[sales['year'] == TARGET_YEAR]['revenue'].sum()
months_completed = sales[sales['year'] == TARGET_YEAR]['month'].nunique()

print(f"Actual revenue so far in {TARGET_YEAR}: {actual_this_year:,.2f}")
print(f"Months completed: {months_completed}")


Actual revenue so far in 2026: 0.00
Months completed: 0


## 5. Project Remaining Months

In [10]:
monthly_revenue = (
    sales.groupby(['year', 'month'], as_index=False)['revenue'].sum()
    .sort_values(['year', 'month'])
)

In [11]:
monthly_revenue

,year,month,revenue
0,2023,1,6282761.26
1,2023,2,5584972.24
2,2023,3,7225276.49
3,2023,4,10616603.50
4,2023,5,12021876.44
5,2023,6,7846609.33
6,2023,7,9589366.48
7,2023,8,11182936.14
8,2023,9,7807263.08
9,2023,10,9224626.01


In [12]:
monthly_revenue['mom_growth'] = monthly_revenue['revenue'].pct_change()

In [13]:
monthly_revenue['mom_growth'].tail(5)

31   -0.031027
32   -0.162807
33    0.109547
34    0.085875
35    0.159465
Name: mom_growth, dtype: float64

In [14]:
avg_growth_rate = monthly_revenue['mom_growth'].tail(12).mean()   # last 12 months ka average

In [15]:
avg_growth_rate

np.float64(0.05854836578180469)

In [16]:
print(f"Average MoM growth rate (last 12 months): {avg_growth_rate*100:.2f}%")

Average MoM growth rate (last 12 months): 5.85%


In [17]:
last_known_revenue = monthly_revenue['revenue'].iloc[-1]
months_remaining = 12 - months_completed

In [18]:
projected_remaining = 0
running_revenue = last_known_revenue

In [19]:
for _ in range(months_remaining):
    running_revenue = running_revenue * (1 + avg_growth_rate)
    projected_remaining += running_revenue

In [20]:
print(f"Months remaining: {months_remaining}")
print(f"Projected revenue for remaining months: {projected_remaining:,.2f}")

Months remaining: 12
Projected revenue for remaining months: 250,633,371.55


## 6. Calculate the Gap

In [21]:
projected_total = actual_this_year + projected_remaining
gap_amount = TARGET_AMOUNT - projected_total
gap_pct = (gap_amount / TARGET_AMOUNT) * 100

In [22]:
print(f"Target:            {TARGET_AMOUNT:,.2f}")
print(f"Projected total:   {projected_total:,.2f}")
print(f"Gap:               {gap_amount:,.2f}  ({gap_pct:.2f}% of target)")

Target:            400,000,000.00
Projected total:   250,633,371.55
Gap:               149,366,628.45  (37.34% of target)


## 7. Required Growth Rate to Hit Target

In [23]:
remaining_target = TARGET_AMOUNT - actual_this_year

if months_remaining > 0:
    required_avg_monthly = remaining_target / months_remaining
    required_growth_rate = (required_avg_monthly / last_known_revenue) - 1
    print(f"Required average monthly revenue (remaining months): {required_avg_monthly:,.2f}")
    print(f"Required MoM growth rate: {required_growth_rate*100:.2f}%")
else:
    required_growth_rate = None
    print("Year already completed (0 months remaining) .")

print(f"Current business growth rate: {avg_growth_rate*100:.2f}%")


Required average monthly revenue (remaining months): 33,333,333.33
Required MoM growth rate: 135.50%
Current business-as-usual growth rate: 5.85%


## 8. Region-Level Breakdown (Which Region Contributes Most to the Gap)

In [24]:
years_with_data = sales['year'].unique()

if TARGET_YEAR in years_with_data and actual_this_year > 0:
    breakdown_year = TARGET_YEAR
else:
    breakdown_year = sales['year'].max()

print(f"Region/category breakdown is using year: {breakdown_year}")

Region/category breakdown is using year: 2025


In [25]:
region_revenue = (
    sales[sales['year'] == breakdown_year]
    .groupby('region', as_index=False)['revenue'].sum()
    .rename(columns={'revenue': 'actual_revenue'})
)

region_revenue['revenue_share'] = region_revenue['actual_revenue'] / region_revenue['actual_revenue'].sum()
region_revenue['pro_rata_target'] = region_revenue['revenue_share'] * TARGET_AMOUNT
region_revenue['shortfall'] = region_revenue['actual_revenue'] - region_revenue['pro_rata_target']

region_revenue = region_revenue.sort_values('shortfall')

print("Region-wise shortfall (most negative = most lagging):")
region_revenue


Region-wise shortfall (most negative = most lagging):


,region,actual_revenue,revenue_share,pro_rata_target,shortfall
1,east,37286287.72,0.291771,1.167082e+08,-7.942193e+07
3,south,25572882.00,0.200111,8.004459e+07,-5.447171e+07
4,west,22417564.24,0.175421,7.016826e+07,-4.775070e+07
2,north,22114832.96,0.173052,6.922070e+07,-4.710586e+07
0,central,20401617.79,0.159646,6.385823e+07,-4.345662e+07


## 9. Category-Level Breakdown

In [31]:
category_revenue = (
    sales[sales['year'] == breakdown_year]
    .groupby('category', as_index=False)['revenue'].sum()
    .rename(columns={'revenue': 'actual_revenue'})
)

category_revenue['revenue_share'] = category_revenue['actual_revenue'] / category_revenue['actual_revenue'].sum()
category_revenue['pro_rata_target'] = category_revenue['revenue_share'] * TARGET_AMOUNT
category_revenue['shortfall'] = category_revenue['actual_revenue'] - category_revenue['pro_rata_target']

category_revenue = category_revenue.sort_values('shortfall')

print("Category-wise shortfall (most negative = most lagging):")
category_revenue


Category-wise shortfall (most negative = most lagging):


,category,actual_revenue,revenue_share,pro_rata_target,shortfall
4,home & kitchen,47963547.91,0.376607,1.506430e+08,-1.026794e+08
5,sports,31261859.84,0.245467,9.818665e+07,-6.692479e+07
3,grocery,17246314.93,0.135417,5.416690e+07,-3.692058e+07
0,apparel,13290389.10,0.104355,4.174220e+07,-2.845181e+07
2,electronics,12135193.89,0.095285,3.811398e+07,-2.597879e+07
1,beauty,5459559.93,0.042868,1.714728e+07,-1.168772e+07


## 10. Build the Final Structured Summary 

In [32]:
gap_summary = {
    "target_amount": TARGET_AMOUNT,
    "target_year": TARGET_YEAR,
    "actual_so_far": round(actual_this_year, 2),
    "projected_total": round(projected_total, 2),
    "gap_amount": round(gap_amount, 2),
    "gap_pct": round(gap_pct, 2),
    "current_growth_rate_pct": round(avg_growth_rate * 100, 2),
    "required_growth_rate_pct": round(required_growth_rate * 100, 2) if required_growth_rate is not None else None,
    "top_underperforming_regions": region_revenue.head(2)[['region', 'shortfall']].to_dict('records'),
    "top_underperforming_categories": category_revenue.head(2)[['category', 'shortfall']].to_dict('records'),
}


In [33]:
import json

In [34]:
print(json.dumps(gap_summary, indent=2))

{
  "target_amount": 400000000,
  "target_year": 2026,
  "actual_so_far": 0.0,
  "projected_total": 250633371.55,
  "gap_amount": 149366628.45,
  "gap_pct": 37.34,
  "current_growth_rate_pct": 5.85,
  "required_growth_rate_pct": 135.5,
  "top_underperforming_regions": [
    {
      "region": "east",
      "shortfall": -79421932.06374969
    },
    {
      "region": "south",
      "shortfall": -54471705.84881942
    }
  ],
  "top_underperforming_categories": [
    {
      "category": "home & kitchen",
      "shortfall": -102679443.13421425
    },
    {
      "category": "sports",
      "shortfall": -66924789.75354965
    }
  ]
}
